# Day 1: Your First SQL Queries

## Objective
Learn the fundamental SQL building blocks: `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`, and basic functions.

## Table of Contents
1. [Connection Setup](#connection-setup)
2. [SELECT Basics](#select-basics)
3. [WHERE Clause - Filtering Data](#where-clause---filtering-data)
4. [ORDER BY - Sorting Results](#order-by---sorting-results)
5. [LIMIT and OFFSET - Pagination](#limit-and-offset---pagination)
6. [DISTINCT - Removing Duplicates](#distinct---removing-duplicates)
7. [Aliases - Renaming Columns](#aliases---renaming-columns)
8. [String Functions](#string-functions)
9. [Date Functions](#date-functions)
10. [Try It Yourself Exercises](#try-it-yourself-exercises)

## Connection Setup

First, let's create a reusable helper function to run SQL queries. This function connects to our PostgreSQL database, executes a query, returns the results as a pandas DataFrame, and always closes the connection.

**Connection parameters explained:**
- `host="localhost"` — The database is running on your local machine
- `port=5432` — PostgreSQL listens on port 5432 (the default)
- `dbname="week2_db"` — The specific database name
- `user="student"` — Your username
- `password="student123"` — Your password

**Why try/finally?** We use `try/finally` to ensure the database connection is **always closed**, even if the query fails. Leaving connections open wastes resources and can cause problems.

In [ ]:
import psycopg2
import pandas as pd

def run_query(sql: str) -> pd.DataFrame:
    """Run a SQL query and return results as a DataFrame."""
    conn = psycopg2.connect(
        host="localhost", port=5432,
        dbname="week2_db", user="student", password="student123"
    )
    try:
        df = pd.read_sql_query(sql, conn)
        return df
    finally:
        conn.close()

print("Helper function defined. Test it with: run_query('SELECT 1 + 1 AS result')")

## SELECT Basics

`SELECT` is how you ask the database for data. It's the first word in almost every SQL query.

**The asterisk (`*`) means "all columns"**: `SELECT *` returns every column in the table.

**`LIMIT` prevents accidentally dumping millions of rows.** Always use `LIMIT` when exploring a new table!

The `company.` prefix specifies the schema — like a folder name. It tells PostgreSQL to look for the `employees` table inside the `company` schema.

In [ ]:
# Get all columns from the first 10 employees
run_query("SELECT * FROM company.employees LIMIT 10")

**Expected output:** A table with 10 rows and 9 columns (emp_id, first_name, last_name, email, department_id, salary, hire_date, manager_id, is_active). You'll see Alice Chen, Bob Martinez, and the first 8 employees.

In [ ]:
# Select only specific columns
run_query("SELECT first_name, salary FROM company.employees LIMIT 10")

**Expected output:** Two columns (first_name, salary) with 10 rows. Only the data we asked for is returned.

## WHERE Clause - Filtering Data

`WHERE` filters rows based on a condition. Only rows where the condition is TRUE are returned.

Let's explore all the comparison operators one by one.

In [ ]:
# = (equals): Find employees in department 1 (Engineering)
run_query("""
    SELECT first_name, last_name, department_id 
    FROM company.employees 
    WHERE department_id = 1
""")

**Expected output:** All employees with department_id = 1 (Alice Chen, Bob Martinez, Carol Johnson, etc.)

In [ ]:
# != or <> (not equals): Employees NOT in department 1
run_query("""
    SELECT first_name, last_name, department_id 
    FROM company.employees 
    WHERE department_id != 1
    LIMIT 10
""")

**Expected output:** Employees from all departments except Engineering.

In [ ]:
# > and < : Employees with salary above 80000
run_query("""
    SELECT first_name, last_name, salary 
    FROM company.employees 
    WHERE salary > 80000
    ORDER BY salary DESC
""")

**Expected output:** High earners: Alice Chen (145000), Nathan Harris (135000), Amy Adams (130000), etc.

In [ ]:
# BETWEEN (inclusive): Salary between 60000 and 90000
run_query("""
    SELECT first_name, last_name, salary 
    FROM company.employees 
    WHERE salary BETWEEN 60000 AND 90000
    ORDER BY salary
""")

**Expected output:** Mid-range earners. BETWEEN includes both endpoints (60000 and 90000).

In [ ]:
# IN: Employees in departments 1, 3, or 5
run_query("""
    SELECT first_name, last_name, department_id 
    FROM company.employees 
    WHERE department_id IN (1, 3, 5)
    ORDER BY department_id, last_name
""")

**Expected output:** Employees from Engineering (1), Finance (3), and Sales (5) departments.

In [ ]:
# LIKE: Pattern matching with strings
# % = any characters, _ = exactly one character
run_query("""
    SELECT first_name, last_name 
    FROM company.employees 
    WHERE first_name LIKE 'A%'  -- Names starting with 'A'
""")

**Expected output:** Alice, Amy (all names starting with A).

In [ ]:
# LIKE with % in the middle: names containing 'an'
run_query("""
    SELECT first_name, last_name 
    FROM company.employees 
    WHERE first_name LIKE '%an%'
""")

**Expected output:** Names like Nathan, etc. — any name with 'an' in it.

### IMPORTANT: IS NULL vs = NULL

**Common beginner trap:** You cannot use `= NULL` to find NULL values. NULL means "unknown," and "unknown = unknown" is not TRUE — it's also unknown!

Always use `IS NULL` or `IS NOT NULL`.

In [ ]:
# IS NULL: Employees with no manager (top-level managers)
run_query("""
    SELECT first_name, last_name, manager_id 
    FROM company.employees 
    WHERE manager_id IS NULL
""")

**Expected output:** Alice Chen, Iris Taylor, Nathan Harris, Uma King, Amy Adams, James Campbell — the directors with no managers above them.

In [ ]:
# WRONG WAY (demonstrating the error!):
# This returns NO rows because = NULL always evaluates to NULL (unknown)
run_query("""
    SELECT first_name, last_name, manager_id 
    FROM company.employees 
    WHERE manager_id = NULL
""")

**Expected output:** Empty table! See? `= NULL` never matches. Always use `IS NULL`.

## ORDER BY - Sorting Results

`ORDER BY` sorts the result set. Default is ascending (`ASC`). Use `DESC` for descending.

In [ ]:
# Sort by salary descending (highest first)
run_query("""
    SELECT first_name, last_name, salary 
    FROM company.employees 
    ORDER BY salary DESC
    LIMIT 10
""")

**Expected output:** Top 10 earners starting with Alice Chen (145000).

In [ ]:
# Sort by department, then by salary within each department
run_query("""
    SELECT first_name, last_name, department_id, salary 
    FROM company.employees 
    ORDER BY department_id ASC, salary DESC
""")

**Expected output:** All employees grouped by department (1, 2, 3, 4, 5, 6), with highest-paid in each department first.

## LIMIT and OFFSET - Pagination

`LIMIT` controls how many rows to return. `OFFSET` controls how many rows to skip.

This is the pattern web apps use for pagination: "Page 3" means `LIMIT 10 OFFSET 20` (skip 20, show next 10).

In [ ]:
# Page 1: First 10 employees
run_query("""
    SELECT first_name, last_name 
    FROM company.employees 
    ORDER BY emp_id
    LIMIT 10 OFFSET 0
""")

**Expected output:** Alice Chen through Iris Taylor (emp_id 1-10).

In [ ]:
# Page 3: Skip 20, get the next 10
run_query("""
    SELECT first_name, last_name 
    FROM company.employees 
    ORDER BY emp_id
    LIMIT 10 OFFSET 20
""")

**Expected output:** Uma King through Henry Turner (emp_id 21-30).

## DISTINCT - Removing Duplicates

`DISTINCT` removes duplicate rows. Only unique combinations are returned.

In [ ]:
# Get unique department IDs (without seeing all employees)
run_query("""
    SELECT DISTINCT department_id 
    FROM company.employees 
    ORDER BY department_id
""")

**Expected output:** Unique department IDs: 1, 2, 3, 4, 5, 6, and NULL (for Paula Morris who has no department).

## Aliases - Renaming Columns

`AS` renames a column in the output. This is especially useful for computed columns.

In [ ]:
# Rename columns and compute annual salary
run_query("""
    SELECT 
        first_name AS name, 
        salary * 12 AS annual_salary
    FROM company.employees 
    LIMIT 10
""")

**Expected output:** Columns named "name" and "annual_salary" (monthly salary × 12).

## String Functions

PostgreSQL has built-in string functions: `UPPER()`, `LOWER()`, `LENGTH()`, `CONCAT()`, and `SUBSTRING()`.

In [ ]:
# UPPER and LOWER
run_query("""
    SELECT 
        first_name,
        UPPER(first_name) AS upper_name,
        LOWER(first_name) AS lower_name
    FROM company.employees 
    LIMIT 5
""")

**Expected output:** first_name alongside uppercase and lowercase versions.

In [ ]:
# LENGTH and CONCAT
run_query("""
    SELECT 
        first_name,
        LENGTH(first_name) AS name_length,
        CONCAT(first_name, ' ', last_name) AS full_name
    FROM company.employees 
    LIMIT 5
""")

**Expected output:** Name length and full name (first + last with space in between).

## Date Functions

PostgreSQL provides powerful date/time functions: `EXTRACT()`, `AGE()`, `NOW()`, and `DATE_TRUNC()`.

In [ ]:
# EXTRACT year from hire_date
run_query("""
    SELECT 
        first_name,
        hire_date,
        EXTRACT(YEAR FROM hire_date) AS hire_year
    FROM company.employees 
    LIMIT 10
""")

**Expected output:** Hire dates alongside the year they were hired.

In [ ]:
# AGE: How long has each employee been with the company?
run_query("""
    SELECT 
        first_name,
        hire_date,
        AGE(hire_date) AS tenure
    FROM company.employees 
    ORDER BY hire_date
    LIMIT 10
""")

**Expected output:** Shows how many years, months, and days each employee has been with the company.

In [ ]:
# DATE_TRUNC: Round a date to the first day of the month
run_query("""
    SELECT 
        hire_date,
        DATE_TRUNC('month', hire_date) AS month_start
    FROM company.employees 
    LIMIT 10
""")

**Expected output:** hire_date and the same date rounded to the first of that month (e.g., 2019-03-15 → 2019-03-01).

## Try It Yourself Exercises

Now it's your turn! Try these exercises. Each one has a hidden solution you can reveal after attempting it yourself.

### Exercise 1
Find all **active** employees hired **after 2023-06-01**, ordered by hire date (most recent first).

<details>
<summary><b>🔍 Click to reveal solution</b></summary>

```sql
SELECT first_name, last_name, hire_date, is_active
FROM company.employees
WHERE is_active = TRUE 
  AND hire_date > '2023-06-01'
ORDER BY hire_date DESC;
```

**Expected output:** Recent hires starting with Paula Morris (2024-07-01), Oscar Stewart, Henry Turner, etc.
</details>

In [ ]:
# Your solution here:
run_query("""
    
""")

### Exercise 2
Find employees whose **last name contains 'son'** (like Johnson, Nelson), showing their full name and salary, ordered by salary descending.

<details>
<summary><b>🔍 Click to reveal solution</b></summary>

```sql
SELECT 
    CONCAT(first_name, ' ', last_name) AS full_name,
    salary
FROM company.employees
WHERE last_name LIKE '%son%'
ORDER BY salary DESC;
```

**Expected output:** Carol Johnson, Cindy Nelson, Eva Patels, etc. with their salaries.
</details>

In [ ]:
# Your solution here:
run_query("""
    
""")

### Exercise 3
Find the **email addresses** of employees who have **no manager** (manager_id IS NULL), showing their first name and email.

<details>
<summary><b>🔍 Click to reveal solution</b></summary>

```sql
SELECT first_name, email
FROM company.employees
WHERE manager_id IS NULL;
```

**Expected output:** The 6 department directors: Alice Chen, Iris Taylor, Nathan Harris, Uma King, Amy Adams, James Campbell.
</details>

In [ ]:
# Your solution here:
run_query("""
    
""")